In [1]:
!pip install requests beautifulsoup4

## Vertex Networking Scraper (v3 — fixed title/description, added price)

**Fix:** their site reused the same generic title/description meta tags on every product page. This version reads the real `<h1>` and the real Description section from the page instead.

**New:** product price is now included.

**Before running:** if you already ran an earlier version and have an existing `vertexnetworking_products.csv` with wrong titles, **delete that CSV file first** — otherwise this version will think those product URLs are already done and skip them (keeping the bad data).

Writes incrementally, resumable, prints progress every 25 products.

In [ ]:
#!/usr/bin/env python3
"""
Scraper for vertexnetworking.co.uk
Walks every category -> subcategory -> product listing (with pagination),
visits each product page, and writes:
    category, image, title, description, price, url
to a CSV file INCREMENTALLY (row-by-row) so no progress is lost if it's
interrupted or crashes partway through.

IMPORTANT FIX vs earlier version:
Their site reuses the SAME <title>/og:title/meta-description on every
product page (a templating bug on their end), so this version reads the
REAL on-page content instead: the <h1> for the title, and the actual
"Description" section/tab for the description. Meta tags are only used
as a last-resort fallback if the on-page elements aren't found.

Requirements:
    pip install requests beautifulsoup4

Usage:
    python scrape_vertexnetworking.py

Output:
    vertexnetworking_products.csv   (written to as it goes)

If you stop the script and run it again later, it will skip any product
URLs already present in the CSV, so it resumes instead of starting over.

If you already have a CSV from the OLD version of this script (with wrong
titles/descriptions), delete or rename it before rerunning, otherwise this
version will think those products are "already done" and skip them.
"""

import csv
import os
import time
import re
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.vertexnetworking.co.uk"
OUTPUT_CSV = "vertexnetworking_products.csv"
REQUEST_DELAY = 0.75   # seconds between requests, be polite to their server
TIMEOUT = 20
PROGRESS_EVERY = 25    # print a running total every N products scraped

FIELDNAMES = ["category", "image", "title", "description", "price", "url"]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

session = requests.Session()
session.headers.update(HEADERS)


def get_soup(url):
    """Fetch a URL and return a BeautifulSoup object, or None on failure."""
    try:
        resp = session.get(url, timeout=TIMEOUT)
        resp.raise_for_status()
        time.sleep(REQUEST_DELAY)
        return BeautifulSoup(resp.text, "html.parser")
    except requests.RequestException as e:
        print(f"  [warn] failed to fetch {url}: {e}")
        return None


def discover_category_links():
    """Find every /category/N and /subcategory/N link from the homepage nav."""
    soup = get_soup(BASE_URL)
    if soup is None:
        raise SystemExit("Could not load homepage — aborting.")

    links = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        full = urljoin(BASE_URL, href)
        if re.search(r"/(category|subcategory)/\d+", full):
            links.add(full)
    return sorted(links)


def get_category_label(soup, fallback_url):
    """Try to pull a readable category/subcategory name from a listing page."""
    h1 = soup.find("h1")
    if h1 and h1.get_text(strip=True):
        return h1.get_text(strip=True)
    title = soup.find("title")
    if title and title.get_text(strip=True):
        return title.get_text(strip=True).split("|")[0].strip()
    return fallback_url


def find_product_links(soup):
    """Extract all product detail page links from a listing page."""
    links = set()
    for a in soup.find_all("a", href=True):
        href = urljoin(BASE_URL, a["href"])
        if "/product/" in href and href.endswith(".html"):
            links.add(href)
    return links


def find_next_page(soup, current_url):
    """Look for a 'next page' pagination link."""
    next_link = soup.find("a", rel="next")
    if next_link and next_link.get("href"):
        return urljoin(current_url, next_link["href"])

    for a in soup.find_all("a", href=True):
        text = a.get_text(strip=True).lower()
        if text in ("next", "»", "next »", ">"):
            return urljoin(current_url, a["href"])

    return None


def clean_description_text(text):
    """Strip repeated 'Description' / 'Product Description' header words that
    get scooped up when we grab the whole section's text."""
    text = re.sub(r"^\s*(Description\s*)+", "", text, flags=re.I)
    text = re.sub(r"^\s*(Product Description\s*)+", "", text, flags=re.I)
    return text.strip()


def scrape_product(url, category_label):
    """Visit a product page and extract title, image, description, price."""
    soup = get_soup(url)
    if soup is None:
        return None

    # ---- TITLE: real <h1> first (meta/og:title on this site is the SAME
    # generic value on every page, so it's only used as a last resort) ----
    title = None
    h1 = soup.find("h1")
    if h1 and h1.get_text(strip=True):
        title = h1.get_text(strip=True)
    if not title:
        og_title = soup.find("meta", property="og:title")
        if og_title and og_title.get("content"):
            title = og_title["content"].strip()

    # ---- IMAGE: real product image first, og:image as fallback ----
    image = None
    img = soup.find("img", src=re.compile(r"/product_images/"))
    if img and img.get("src"):
        image = urljoin(url, img["src"])
    if not image:
        og_image = soup.find("meta", property="og:image")
        if og_image and og_image.get("content"):
            image = urljoin(url, og_image["content"].strip())

    # ---- DESCRIPTION: the real "Description" section/tab on the page
    # first, meta description as a last-resort fallback ----
    description = None
    desc_section = soup.find(["div", "section"], id=re.compile(r"description", re.I))
    if not desc_section:
        desc_section = soup.find(
            ["div", "section"],
            class_=re.compile(r"(product-desc|description|details)", re.I),
        )
    if desc_section:
        text = desc_section.get_text(" ", strip=True)
        text = clean_description_text(text)
        if text:
            description = text
    if not description:
        meta_desc = soup.find("meta", attrs={"name": "description"})
        if meta_desc and meta_desc.get("content"):
            description = meta_desc["content"].strip()

    # ---- PRICE ----
    price = None
    price_tag = soup.find(["span", "div", "p"], class_=re.compile(r"price", re.I))
    if price_tag:
        m = re.search(r"[\$£€]\s?[\d,]+\.?\d*", price_tag.get_text(" ", strip=True))
        if m:
            price = m.group(0).strip()
    if not price:
        # fallback: scan the whole page text for the first currency-looking value
        m = re.search(r"[\$£€]\s?[\d,]+\.\d{2}", soup.get_text(" ", strip=True))
        if m:
            price = m.group(0).strip()

    return {
        "category": category_label,
        "image": image or "",
        "title": title or "",
        "description": description or "",
        "price": price or "",
        "url": url,
    }


def load_already_scraped():
    """If the CSV already exists (from a previous run), load which product
    URLs are already done so we can skip them and resume."""
    done = set()
    if os.path.exists(OUTPUT_CSV):
        with open(OUTPUT_CSV, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row.get("url"):
                    done.add(row["url"])
    return done


def open_csv_writer():
    """Open the CSV in append mode, writing the header only if new."""
    file_exists = os.path.exists(OUTPUT_CSV)
    f = open(OUTPUT_CSV, "a", newline="", encoding="utf-8")
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    if not file_exists:
        writer.writeheader()
        f.flush()
    return f, writer


def main():
    already_done = load_already_scraped()
    if already_done:
        print(f"Resuming — {len(already_done)} products already in {OUTPUT_CSV}, will skip those.")

    csv_file, writer = open_csv_writer()
    scraped_count = len(already_done)

    try:
        print("Discovering category/subcategory links...")
        category_links = discover_category_links()
        print(f"Found {len(category_links)} category/subcategory links.")

        for cat_index, cat_url in enumerate(category_links, start=1):
            print(f"\n[{cat_index}/{len(category_links)}] Category page: {cat_url}")
            page_url = cat_url
            visited_pages = set()

            while page_url and page_url not in visited_pages:
                visited_pages.add(page_url)
                soup = get_soup(page_url)
                if soup is None:
                    break

                label = get_category_label(soup, cat_url)
                product_links = find_product_links(soup)
                new_links = [p for p in product_links if p not in already_done]
                print(f"  {page_url} -> {len(product_links)} product links "
                      f"({len(new_links)} new)")

                for p_url in new_links:
                    row = scrape_product(p_url, label)
                    already_done.add(p_url)
                    if row:
                        writer.writerow(row)
                        csv_file.flush()  # write to disk immediately
                        scraped_count += 1
                        if scraped_count % PROGRESS_EVERY == 0:
                            print(f"  >>> progress: {scraped_count} products saved so far")

                page_url = find_next_page(soup, page_url)

    except KeyboardInterrupt:
        print("\n[stopped by user] Progress so far is safely saved in the CSV.")
    finally:
        csv_file.close()

    print(f"\nTotal products in {OUTPUT_CSV}: {scraped_count}")
    print("Done (or safely stopped — rerun the script anytime to resume).")


if __name__ == "__main__":
    main()


Discovering category/subcategory links...
Found 120 category/subcategory links.

[1/120] Category page: https://www.vertexnetworking.co.uk/category/1
  https://www.vertexnetworking.co.uk/category/1 -> 20 product links (20 new)
  https://www.vertexnetworking.co.uk/category/1?page=2 -> 20 product links (20 new)
  >>> progress: 25 products saved so far
  https://www.vertexnetworking.co.uk/category/1?page=3 -> 20 product links (20 new)
  >>> progress: 50 products saved so far
  https://www.vertexnetworking.co.uk/category/1?page=4 -> 20 product links (20 new)
  >>> progress: 75 products saved so far
  https://www.vertexnetworking.co.uk/category/1?page=5 -> 20 product links (20 new)
